# Calibrate using TensorFlow

In [ ]:
import os
import numpy as np
import xarray as xr
import pandas as pd
import dask
from dask_jobqueue import PBSCluster
from dask.distributed import Client

import fates_calibration_library.emulator_functions as em
import fates_calibration_library.utils as utils
from fates_calibration_library.TFClass import TFEmulator


import tensorflow as tf
import tensorflow_probability as tfp

import importlib

from esem import gp_model

import gpflow

## Set Up
Load files, set up ensemble information

In [ ]:
# directories
mesh_dir = '/glade/work/afoster/FATES_calibration/mesh_files'
emulator_dir = '/glade/work/afoster/FATES_calibration/emulators'
fig_dir = '/glade/work/afoster/FATES_calibration/figures'
param_dir = '/glade/work/afoster/FATES_calibration/parameter_files'

# default parameter file
default_param = xr.open_dataset(os.path.join(param_dir, 
                                             'fates_params_default_sci.1.85.1_api.40.0.0_crops.nc'))
all_pfts = [str(pft).replace("b'", "").replace("'", "").strip() for pft in default_param.fates_pftname.values]

# normalized values for parameters
default_norm = pd.read_csv(os.path.join(param_dir, 'normalized_parameters.csv'), index_col=[0])

# variables to calibrate
calibration_vars = ['GPP', 'EFLX_LH_TOT', 'FSH', 'EF']

obs_config_file = '/glade/work/afoster/FATES_calibration/fates_calibration_library/configs/ilamb_conversion.yaml'
obs_config = utils.get_config_file(obs_config_file)

In [ ]:
# information about each ensemble
ens_dict = {'dompft':
            {'mesh_file': os.path.join(mesh_dir, 'dominant_grid_mesh.nc'),
             'land_mask_file': os.path.join(mesh_dir, 'dominant_grid.nc'),
             'lhc_key_file': os.path.join(param_dir, 'fates_lh', 'fates_lh_key.csv'),
             'pfts': [1, 2, 3, 12, 13, 14],
             'obs_df': os.path.join(mesh_dir, 'dominant_grid.csv'),
            }
           }

In [ ]:
# choose ensemble
ensemble = 'dompft'

### Load Latin Hypercube Key

In [ ]:
lhc_key = pd.read_csv(ens_dict[ensemble]['lhc_key_file'], index_col=[0])
lhc_key = lhc_key.drop(columns=['ensemble'])
param_names = lhc_key.columns
num_params = len(param_names)

### Load Observations

In [ ]:
obs = pd.read_csv(ens_dict[ensemble]['obs_df'], index_col=[0])

## Calibration

In [ ]:
class TFEmulator:
    def __init__(self, model_dir, pft, variable):
        path = os.path.join(model_dir, f"{pft}_{variable}")
        self.loaded = tf.saved_model.load(path)
        self.predict_fn = self.loaded.signatures["serving_default"]

    def __call__(self, X):
        X_tensor = tf.convert_to_tensor(X, dtype=tf.float64)
        output = self.predict_fn(X=X_tensor)
        return output["mean"], output["variance"]

In [ ]:
pft = 1
pft_name = all_pfts[pft-1]

In [ ]:
# get observations for this pft
obs_pft = obs[obs.pft == pft_name]

# get default values for this pft
default_pft = default_norm[default_norm.pft == pft]
default_pft = default_pft.drop(columns = ['pft'])

# convert to tf object
x_default = tf.constant(default_pft.to_numpy(), dtype=tf.float64)

In [ ]:
# stack targets, sds, and emulators for all variables
targets = []
sds = []
emulators = []
for variable in calibration_vars:
    
    # observations for this pft and variable
    obs_mean, obs_sd = em.get_obs_mean_and_sd(obs_pft, obs_config[variable]['var'])
    
    # convert to tf objects
    targets.append(tf.convert_to_tensor(obs_mean, dtype=tf.float64))
    sds.append(tf.convert_to_tensor(obs_sd, dtype=tf.float64))

    # load the emulator
    emulators.append(TFEmulator(emulator_dir, pft=pft_name, variable=variable))

In [ ]:
from SALib.sample import fast_sampler
from SALib.analyze import fast

In [ ]:
# create a fast sample for fourier sensitivity
problem = {
    'names': param_names,
    'num_vars': len(param_names),
    'bounds': [[0, 1]],
}

In [ ]:
sample = fast_sampler.sample(problem, 1000, M=4, seed=None)

In [ ]:
emulator = emulators[0]

In [ ]:
Y, _ = emulator(sample)

In [ ]:
FAST = fast.analyze(problem, Y.numpy(), M=4, num_resamples=100, conf_level=0.95,
                    print_to_console=False, seed=None)

In [ ]:
sens = pd.DataFrame.from_dict(FAST)
sens.index = sens.names

In [ ]:
df_sens = sens.sort_values(by=['S1'], ascending=False)

In [ ]:
df_sens

In [ ]:
importlib.reload(em)

In [ ]:
config = {
    'checkpoint_dir': '/glade/work/afoster/FATES_calibration/checkpoints',
    'learning_rate': 1e-3,
    'lr_decay_steps': 300,
    'maxiter': 3000,
    'checkpoint_n': 10,
    'epsilon': 0.5,
    'lambda_penalty': None,
    'barrier_strength': 0,
    'earlystop_pct': 90.0,
    'loss_fn': em.implausibility_loss,
    'default_penalty_fn': em.default_penalty_l1,
    'barrier_penalty_fn': em.barrier_penalty,
}

In [ ]:
X = tf.Variable(x_default, dtype=tf.float64, trainable=True, name = 'X')

In [ ]:
X = tf.Variable(tf.random.uniform(shape=(1000, num_params),
                                  minval=0.0,
                                  maxval=1.0,
                                  dtype=tf.float64),
                dtype=tf.float64, trainable=True, name='X')

In [ ]:
X_opt, logs = em.run_optimization(X, emulators, targets, sds, x_default, config)

In [ ]:
np.argmin(logs['losses'][2999])

In [ ]:
X_opt[930]

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(logs['total_loss'], label='Total loss')
plt.plot(logs['data_loss'], label='Data loss')
plt.plot(logs['default_penalty'], label='Penalty loss')
plt.plot(logs['barrier_penalty'], label='Barrier loss')
plt.xlabel('Step')
plt.ylabel('Loss value')
plt.legend()
plt.grid(True)
plt.title('Loss components over optimization steps')
#plt.ylim(0,5)
#plt.savefig('loss_plot_unifRandom_minError_clipNoPenalties.png')

In [ ]:
calibrated_params = pd.DataFrame(X_opt, columns=param_names)

In [ ]:
param_names

In [ ]:
plt.hist(calibrated_params[param_names[3]])

In [ ]:
plt.scatter(calibrated_params[param_names[7]], calibrated_params[param_names[20]])

In [ ]:
param_names[14]